# 01. Data Understanding, Profiling & Schema Architecture
**Project:** Telco Customer Churn & Retention Intelligence  
**Phase:** Phase 2 - Exploratory Data Analysis & Customer Behavior Analysis  
**Dataset:** `data/processed/telco_churn_cleaned.csv`

---

## Executive Overview & Objectives
This notebook establishes foundational data understanding and schema validation for the Telco Customer Churn dataset. We inspect:
1. **Dataset dimensions** and schema architecture.
2. **Data types, missing values**, and data hygiene.
3. **Primary key uniqueness** and customer identity verification.
4. **Target variable distribution** (`Churn`) and class balance.
5. **Numerical feature distributions** (`tenure`, `MonthlyCharges`, `TotalCharges`).
6. **Categorical feature taxonomy** and attribute cardinalities.


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.autolayout'] = True

DATA_PATH = (Path.cwd().parent / "data/processed/telco_churn_cleaned.csv" 
             if Path.cwd().name == "notebooks" 
             else Path("data/processed/telco_churn_cleaned.csv")).resolve()
FIGURES_DIR = (Path.cwd().parent / "reports/figures" 
               if Path.cwd().name == "notebooks" 
               else Path("reports/figures")).resolve()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded from: {DATA_PATH.resolve()}")
print(f"Total Rows (Customers): {df.shape[0]:,}")
print(f"Total Columns (Features): {df.shape[1]}")


Dataset successfully loaded from: C:\Users\Sruthi\Documents\telco-churn-app\data\processed\telco_churn_cleaned.csv
Total Rows (Customers): 7,043
Total Columns (Features): 21


## 1. Schema & Feature Taxonomy
The dataset contains 21 features capturing the full customer lifecycle:
- **Identifier:** `customerID`
- **Customer Demographics:** `gender`, `SeniorCitizen`, `Partner`, `Dependents`
- **Account & Tenure:** `tenure`
- **Telephony Services:** `PhoneService`, `MultipleLines`
- **Internet & Security Services:** `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`
- **Streaming Entertainment:** `StreamingTV`, `StreamingMovies`
- **Contract & Billing:** `Contract`, `PaperlessBilling`, `PaymentMethod`
- **Financial Charges:** `MonthlyCharges`, `TotalCharges`
- **Target Variable:** `Churn`


In [2]:
schema_info = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Non-Null Count": df.notnull().sum(),
    "Null Count": df.isnull().sum(),
    "Null %": (df.isnull().sum() / len(df) * 100).round(2),
    "Unique Values": df.nunique()
})
print("--- Data Schema & Completeness ---")
print(schema_info.to_string())


--- Data Schema & Completeness ---
                 Data Type  Non-Null Count  Null Count  Null %  Unique Values
customerID          object            7043           0     0.0           7043
gender              object            7043           0     0.0              2
SeniorCitizen        int64            7043           0     0.0              2
Partner             object            7043           0     0.0              2
Dependents          object            7043           0     0.0              2
tenure               int64            7043           0     0.0             73
PhoneService        object            7043           0     0.0              2
MultipleLines       object            7043           0     0.0              3
InternetService     object            7043           0     0.0              3
OnlineSecurity      object            7043           0     0.0              3
OnlineBackup        object            7043           0     0.0              3
DeviceProtection    object   

## 2. Customer ID Integrity & Primary Key Audit
Verifying primary key uniqueness guarantees that records represent distinct individual accounts and avoids double-counting revenue or churn events.


In [3]:
total_rows = len(df)
unique_customers = df['customerID'].nunique()
duplicate_count = df.duplicated(subset=['customerID']).sum()
null_customers = df['customerID'].isna().sum()

print("--- Primary Key Integrity Check ---")
print(f"Total Table Records    : {total_rows}")
print(f"Unique Customer IDs    : {unique_customers}")
print(f"Duplicate Customer IDs : {duplicate_count}")
print(f"Null Customer IDs      : {null_customers}")

assert total_rows == unique_customers, "Duplicate customer IDs detected!"
assert duplicate_count == 0, "Duplicate rows detected!"
print("Status: Primary key constraint verified. Every row represents an independent customer account.")


--- Primary Key Integrity Check ---
Total Table Records    : 7043
Unique Customer IDs    : 7043
Duplicate Customer IDs : 0
Null Customer IDs      : 0
Status: Primary key constraint verified. Every row represents an independent customer account.


## 3. Target Variable Analysis (`Churn`)
We evaluate the class distribution of `Churn`. Churn indicates whether a customer left the telecom company within the last billing month (`Yes` vs. `No`).


In [4]:
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    "Customer Count": churn_counts,
    "Percentage (%)": churn_percentages.round(2)
})
print("--- Churn Class Distribution ---")
print(churn_summary)

# Visualization: Target Distribution (Bar Chart + Donut Chart)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2b5c8f', '#d9534f']
bars = axes[0].bar(churn_summary.index, churn_summary['Customer Count'], color=colors, width=0.5, edgecolor='black')
axes[0].set_title("Customer Churn Volume (Snapshot)", fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel("Churn Status", fontsize=12)
axes[0].set_ylabel("Number of Customers", fontsize=12)
axes[0].set_ylim(0, 6000)
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 100, f"{height:,} ({height/len(df)*100:.1f}%)",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

axes[1].pie(churn_summary['Customer Count'], labels=churn_summary.index, autopct='%1.2f%%',
            startangle=90, colors=colors, wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title("Proportion of Churned vs. Active Customers", fontsize=14, fontweight='bold', pad=12)

out_fig = FIGURES_DIR / "01_churn_distribution.png"
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {out_fig.resolve()}")


--- Churn Class Distribution ---
       Customer Count  Percentage (%)
Churn                                
No               5174           73.46
Yes              1869           26.54
[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\01_churn_distribution.png


## 4. Numerical Features Statistical Profiling
The dataset includes three primary numerical variables:
1. `tenure`: Number of months the customer has stayed with the company (0 to 72 months).
2. `MonthlyCharges`: Current recurring monthly charges (USD).
3. `TotalCharges`: Cumulative charges over the customer's lifespan (USD).


In [5]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

stats_df = df[num_cols].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).T
stats_df['IQR'] = stats_df['75%'] - stats_df['25%']
stats_df['Skewness'] = df[num_cols].skew()
stats_df['Kurtosis'] = df[num_cols].kurtosis()
print(stats_df.round(2).to_string())


                 count     mean      std    min     5%     25%      50%      75%      95%      max      IQR  Skewness  Kurtosis
tenure          7043.0    32.37    24.56   0.00   1.00    9.00    29.00    55.00    72.00    72.00    46.00      0.24     -1.39
MonthlyCharges  7043.0    64.76    30.09  18.25  19.65   35.50    70.35    89.85   107.40   118.75    54.35     -0.22     -1.26
TotalCharges    7043.0  2279.73  2266.79   0.00  48.60  398.55  1394.55  3786.60  6921.02  8684.80  3388.05      0.96     -0.23


In [6]:
# Visualizing Numerical Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#1f77b4', edgecolor='black', alpha=0.6)
    axes[i].set_title(f"Distribution of {col}", fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col, fontsize=11)
    axes[i].set_ylabel("Customer Count", fontsize=11)
    
    mean_val = df[col].mean()
    median_val = df[col].median()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f"Mean: {mean_val:.1f}")
    axes[i].axvline(median_val, color='green', linestyle='-', linewidth=1.5, label=f"Median: {median_val:.1f}")
    axes[i].legend(loc='upper right')

out_fig2 = FIGURES_DIR / "01_numerical_distributions.png"
plt.savefig(out_fig2, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {out_fig2.resolve()}")


[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\01_numerical_distributions.png


## 5. Categorical Feature Distributions & Cardinalities
We review the value counts for each categorical feature to inspect cardinalities across subscriber categories.


In [7]:
cat_cols = [c for c in df.columns if c not in num_cols + ['customerID', 'Churn']]

cat_profile = []
for c in cat_cols:
    val_counts = df[c].value_counts().to_dict()
    cat_profile.append({
        "Feature": c,
        "Distinct Values": len(val_counts),
        "Breakdown": val_counts
    })

pd.set_option('display.max_colwidth', None)
cat_df = pd.DataFrame(cat_profile)
print(cat_df.to_string(index=False))


         Feature  Distinct Values                                                                                                            Breakdown
          gender                2                                                                                       {'Male': 3555, 'Female': 3488}
   SeniorCitizen                2                                                                                                   {0: 5901, 1: 1142}
         Partner                2                                                                                            {'No': 3641, 'Yes': 3402}
      Dependents                2                                                                                            {'No': 4933, 'Yes': 2110}
    PhoneService                2                                                                                             {'Yes': 6361, 'No': 682}
   MultipleLines                3                                                             

## 6. Key Findings & Data Understanding Summary
1. **Customer Base:** Exactly 7,043 unique customer accounts with zero duplicate IDs and zero missing values in the cleaned table.
2. **Churn Baseline:** 1,869 customers churned (26.54%), representing a moderate class imbalance (~1:3 ratio).
3. **Tenure Bimodality:** `tenure` exhibits an extreme bimodal distribution with spikes at month 1 (new signups) and month 72 (long-term loyalists).
4. **Charge Skewness:** `TotalCharges` is heavily right-skewed (skewness = +0.96) reflecting tenure variation, while `MonthlyCharges` exhibits a multi-modal distribution peaking at ~$20 (basic phone) and ~$80–$90 (fiber optic + streaming bundles).
